# Check dataset for privacy audits
In order to run this script, first load data with DataLoad.ipynb. It reads parquet and converts to Delta Table for more efficient loading and filtering. 

The checks are executed on the 5-min aggregates in Delta Table histograms
 
When there can be a difference from 1-min aggregates, such as time information, then parquet files are checked. Sample of parquet files is shared by request and certain conditions depending on ISP.

In [ ]:
from delta import DeltaTable
import datetime as D
from pyspark.sql import functions as F
import pyspark.sql

from IPython.display import display, HTML

# parquet_src_path = "add your parquet path here"
delta_db_path = "add your delta path here"

# Initiate spark
spark = "get your spark session here"
sc = spark.sparkContext

# load data
delta_table = DeltaTable.forPath(spark, delta_db_path)
df0 = delta_table.toDF()

## Statement: No IPs exist in dataset

In [7]:
df0.printSchema()

root
 |-- date: date (nullable = true)
 |-- creator: string (nullable = true)
 |-- label0: string (nullable = true)
 |-- label1: string (nullable = true)
 |-- label2: string (nullable = true)
 |-- label3: string (nullable = true)
 |-- label4: string (nullable = true)
 |-- label5: string (nullable = true)
 |-- label6: string (nullable = true)
 |-- label7: string (nullable = true)
 |-- label8: string (nullable = true)
 |-- label9: string (nullable = true)
 |-- hour: byte (nullable = true)
 |-- minute: byte (nullable = true)
 |-- tagstring: string (nullable = true)
 |-- fqdn: string (nullable = true)
 |-- r_fqdn: string (nullable = true)
 |-- idn_fqdn: string (nullable = true)
 |-- a_count: long (nullable = true)
 |-- aaaa_count: long (nullable = true)
 |-- mx_count: long (nullable = true)
 |-- ns_count: long (nullable = true)
 |-- other_type_count: long (nullable = true)
 |-- non_in_count: long (nullable = true)
 |-- ok_count: long (nullable = true)
 |-- nx_count: long (nullable = true)


## Contents of TAPIR Core aggregates 

In [9]:
creator = "competent-albattani.test.dnstapir.se"
df = df0.where((df0.creator == creator) & (df0.date == "2026-08-20") & (df0.hour == 10))
df.toPandas()

,date,creator,label0,label1,label2,label3,label4,label5,label6,label7,...,other_type,non_in,v4_clients,v6_clients,v4clients_hll,v6clients_hll,v4clients_avg,v6clients_avg,v4client_count_hll,v6client_count_hll
0,2026-08-20,competent-albattani.test.dnstapir.se,com,live,account,None,None,None,None,None,...,[0],[0],[1],[0],"[17, 106, 127]","[17, 106, 127]",1.000000,0.000000,0,0
1,2026-08-20,competent-albattani.test.dnstapir.se,com,google,accounts,None,None,None,None,None,...,"[1, 0, 2]","[0, 0, 0]","[1, 2, 4]","[0, 0, 1]","[17, 106, 127]","[17, 106, 127]",2.333333,0.333333,0,0
2,2026-08-20,competent-albattani.test.dnstapir.se,com,spotify,accounts,None,None,None,None,None,...,[0],[0],[1],[0],"[17, 106, 127]","[17, 106, 127]",1.000000,0.000000,0,0
3,2026-08-20,competent-albattani.test.dnstapir.se,net,doubleclick,ad,None,None,None,None,None,...,"[0, 1, 0, 2, 1]","[0, 0, 0, 0, 0]","[1, 1, 1, 1, 1]","[0, 0, 0, 2, 0]","[17, 106, 127]","[17, 106, 127]",1.000000,0.400000,0,0
4,2026-08-20,competent-albattani.test.dnstapir.se,com,adobedtm,assets,None,None,None,None,None,...,[1],[0],[1],[1],"[17, 106, 127]","[17, 106, 127]",1.000000,1.000000,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1549,2026-08-20,competent-albattani.test.dnstapir.se,com,yahoo,search,None,None,None,None,None,...,[0],[0],[1],[0],"[17, 106, 127]","[17, 106, 127]",1.000000,0.000000,0,0
1550,2026-08-20,competent-albattani.test.dnstapir.se,net,doubleclick,g,securepubads,None,None,None,None,...,"[0, 0, 0]","[0, 0, 0]","[1, 1, 1]","[0, 1, 0]","[17, 106, 127]","[17, 106, 127]",1.000000,0.333333,0,0
1551,2026-08-20,competent-albattani.test.dnstapir.se,com,google-analytics,ssl,None,None,None,None,None,...,[0],[0],[1],[0],"[17, 106, 127]","[17, 106, 127]",1.000000,0.000000,0,0
1552,2026-08-20,competent-albattani.test.dnstapir.se,com,sway,None,None,None,None,None,None,...,[0],[0],[1],[0],"[17, 106, 127]","[17, 106, 127]",1.000000,0.000000,0,0


## Look for IP formatted string in columns with string datatype

- IP addresses can also be stored as <> datatype but as shown in schema above, no such datatype exists. The technically correct way to store IPv4 is binary(4) ?  but it can also be stored as strings
- Simple check if col value in sample matches IPv4, IPv6 pattern.
- It's known that such formats can exist in domain names from cdns and in-addr.arpa, ip6.arpa. Currently these are wildcarded away in TAPIR Core, and even if not, that's the domain owner's data responsibility. In case domain names with IPs are found, that are not widely known, this could be reported to domain owner.

**Sample of 5-min aggregates, string columns**

In [10]:
string_cols = [name for name, dtype in df0.dtypes if dtype == "string"]
df0.select(string_cols).limit(5).toPandas()

,creator,label0,label1,label2,label3,label4,label5,label6,label7,label8,label9,tagstring,fqdn,r_fqdn,idn_fqdn
0,compassionate-albattani.test.dnstapir.se,com,googleusercontent,ci3,None,None,None,None,None,None,None,A,ci3.googleusercontent.com.,com.googleusercontent.ci3.,None
1,compassionate-albattani.test.dnstapir.se,com,gstatic,fonts,None,None,None,None,None,None,None,A,fonts.gstatic.com.,com.gstatic.fonts.,None
2,compassionate-albattani.test.dnstapir.se,com,google,inbox,None,None,None,None,None,None,None,A,inbox.google.com.,com.google.inbox.,None
3,compassionate-albattani.test.dnstapir.se,com,googleusercontent,lh3,None,None,None,None,None,None,None,A,lh3.googleusercontent.com.,com.googleusercontent.lh3.,None
4,compassionate-albattani.test.dnstapir.se,com,googleapis,oauth2,None,None,None,None,None,None,None,A,oauth2.googleapis.com.,com.googleapis.oauth2.,None


In [11]:
# small sample to illustrate, change when needed
check_date = "2026-08-18"
df_sample = df0.where((df0.creator == creator) & (df0.date == check_date)).limit(100)

In [ ]:
# Check ipv4
for col in string_cols:
    count = df_sample.where(
        F.col(col).rlike(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}")
    ).count()
    print(f"{col}: {count} IPv4-liknande värden")

creator: 0 IPv4-liknande värden
label0: 0 IPv4-liknande värden
label1: 0 IPv4-liknande värden
label2: 0 IPv4-liknande värden
label3: 0 IPv4-liknande värden
label4: 0 IPv4-liknande värden
label5: 0 IPv4-liknande värden
label6: 0 IPv4-liknande värden
label7: 0 IPv4-liknande värden
label8: 0 IPv4-liknande värden
label9: 0 IPv4-liknande värden
tagstring: 0 IPv4-liknande värden
fqdn: 0 IPv4-liknande värden
r_fqdn: 0 IPv4-liknande värden
idn_fqdn: 0 IPv4-liknande värden


In [25]:
# Check ipv6

# This catches some formats of ipv6. Not misspelled etc.
# ipv6 could aso use _ or - not catched here
# Add when needed.


import re

ex_ipv6 = "2001:0db8:85a3:0000:0000:8a2e:0370:7334"
ex_ipv6_short = "2001:db8:85a3::8a2e:370:7334"
ex_ipv6_nibble = "5.b.3.2.3.7.6.5.7.6.5.0.0.0.0.0.0.0.0.0.1.a.c.2.5.2.4.0.5.4.3.2"


# base pattern: # regexp = r"([0-9a-f]{1,4}:){7}[0-9a-f]{1,4}"

ipv6_pattern = (
    r"([0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}|"
    r"([0-9a-fA-F]{1,4}:){1,7}:|"
    r"([0-9a-fA-F]{1,4}:){1,6}:[0-9a-fA-F]{1,4}|"
    r"([0-9a-fA-F]{1,4}:){1,5}(:[0-9a-fA-F]{1,4}){1,2}|"
    r"([0-9a-fA-F]{1,4}:){1,4}(:[0-9a-fA-F]{1,4}){1,3}|"
    r"([0-9a-fA-F]{1,4}:){1,3}(:[0-9a-fA-F]{1,4}){1,4}|"
    r"([0-9a-fA-F]{1,4}:){1,2}(:[0-9a-fA-F]{1,4}){1,5}|"
    r"[0-9a-fA-F]{1,4}:(:[0-9a-fA-F]{1,4}){1,6}|"
    r":(:[0-9a-fA-F]{1,4}){1,7}|"
    r"::"
)

ipv6_nibble_pattern = r"[0-9a-fA-F](\.[0-9a-fA-F]){31}"


# regexp = r"([0-9a-f]{4}:){7}[0-9a-f]{4}"
# regexp = r":"

# print(regexp_cs.search(ex_ipv6))
# print(regexp_c.search(ex_ipv6_short))


<re.Match object; span=(0, 15), match='2001:db8:85a3::'>


In [ ]:
regexp_c = re.compile(ipv6_pattern)
regexp_c2 = re.compile(ipv6_nibble_pattern)

string_cols = [name for name, dtype in df0.dtypes if dtype == "string"]

for col in string_cols:
    df_colon = df_sample.where(F.col(col).rlike(r":"))
    colon_count = df_colon.count()
    if colon_count > 0:
        ipv6_count = df_colon.where(F.col(col).rlike(ipv6_pattern)).count()
        print(f"{col}: {colon_count} with colon, {ipv6_count} matching IPv6 pattern")
    else:
        print(f"{col}: 0 colon matches")

    nibble_count = df_sample.where(F.col(col).rlike(ipv6_nibble_pattern)).count()
    if nibble_count > 0:
        print(f"{col}: {nibble_count} matching IPv6 nibble pattern")
    else:
        print(f"{col}: 0 nibble matches")

creator: 0 colon matches
creator: 0 nibble matches
label0: 0 colon matches
label0: 0 nibble matches
label1: 0 colon matches
label1: 0 nibble matches
label2: 0 colon matches
label2: 0 nibble matches
label3: 0 colon matches
label3: 0 nibble matches
label4: 0 colon matches
label4: 0 nibble matches
label5: 0 colon matches
label5: 0 nibble matches
label6: 0 colon matches
label6: 0 nibble matches
label7: 0 colon matches
label7: 0 nibble matches
label8: 0 colon matches
label8: 0 nibble matches
label9: 0 colon matches
label9: 0 nibble matches
tagstring: 0 colon matches
tagstring: 0 nibble matches
fqdn: 0 colon matches
fqdn: 0 nibble matches
r_fqdn: 0 colon matches
r_fqdn: 0 nibble matches
idn_fqdn: 0 colon matches
idn_fqdn: 0 nibble matches


## No exact timestamps exists in aggregates
- In this case exact timestamp is hour, minute, second, millisecond